# Market Potential Predictor — End-to-End Walkthrough

This notebook walks through a single district from raw data to opportunity flag.

In [ ]:
import geopandas as gpd
from market_predictor.config import AppConfig
from market_predictor.data.mock import generate_mock_data
from market_predictor.data.features import add_synthetic_features
from market_predictor.pipeline.zonal import extract_zonal_statistics
from market_predictor.pipeline.scoring import calculate_market_potential_score
from market_predictor.pipeline.opportunity import identify_high_potential_zones

cfg = AppConfig.load()
generate_mock_data(cfg.raster_path, cfg.districts_path)

In [ ]:
districts = gpd.read_file(cfg.districts_path)
district = districts.iloc[0]
print(f"District: {district['district_id']}")
print(f"Population density: {district['population_density']}")
print(f"Current businesses: {district['current_business_count']}")

In [ ]:
zonal = extract_zonal_statistics(districts, cfg.raster_path)
zonal = add_synthetic_features(zonal)
row = zonal.iloc[0]
print(f"Mean night light: {row['mean_night_light']}")
print(f"Median income: {row['median_income']}")
print(f"Road access: {row['road_access_score']}")

In [ ]:
scored = calculate_market_potential_score(zonal)
result = identify_high_potential_zones(scored)
row = result.iloc[0]
print(f"Market Potential Score: {row['market_potential_score']:.4f}")
print(f"High-potential untapped: {row['is_high_potential_untapped']}")

In [ ]:
flagged = result[result['is_high_potential_untapped']].sort_values('opportunity_rank')
flagged[['district_id', 'market_potential_score', 'current_business_count', 'opportunity_rank']].head(10)